# Synthèse de la recherche de règles

Lit les sorties de `src/recherche_regles.ipynb`, après vérification de leurs empreintes, et produit les chiffres repris dans `research/resultats_recherche.md` qui ne sont pas déjà imprimés par ce carnet.

In [1]:
import sys, json, gzip
from pathlib import Path
import numpy as np
import pandas as pd
RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
from src.empreintes import correspond
SORTIE = RACINE / "data" / "processed"
m = json.loads((SORTIE / "regles_manifest.json").read_text(encoding="utf-8"))
for nom, sha in m["sorties_sha256"].items():
    assert correspond(SORTIE / nom, sha), nom
t = pd.read_csv(SORTIE / "regles_mesures.csv", index_col=0)
hasard = pd.read_csv(SORTIE / "aleatoires_fenetre_2011.csv.gz")
gagnant = m["controles"]["gagnant"]
print("gagnant :", gagnant)
print(t.loc[gagnant, ["annualise", "volatilite", "repli_max", "es99", "sharpe", "beta_spy", "centile_sharpe"]].round(4).to_string())
print()
print("taux d'acceptation par famille de protection :")
print(t.groupby("famille_protection").acceptable.mean().round(3).to_string())
print()
print("regles acceptables sans protection :", list(t[(t.protection == "P0") & t.acceptable].index))
print()
print("centile de Sharpe median des regles, par selection :")
print(t.groupby("selection").centile_sharpe.median().round(3).to_string())
print()
print("portefeuilles aleatoires au-dessus du Sharpe de T1S (0,8604) : %.1f %%" % (100 * (hasard.sharpe > 0.8604).mean()))
print("portefeuilles aleatoires qui satisfont les trois criteres face a T1S : %.1f %%" % (100 * (
    (hasard.repli_max >= -0.3821) & (hasard.es99 <= 0.0442) & (hasard.annualise >= 0.1609)).mean()))
print("Sharpe du gagnant face aux portefeuilles aleatoires : superieur a %.1f %% d'entre eux" % (100 * (hasard.sharpe < t.loc[gagnant, "sharpe"]).mean()))
print()
print("meilleure regle par famille de protection :")
print(t.sort_values("sharpe", ascending=False).groupby("famille_protection").head(1)[["annualise", "volatilite", "repli_max", "es99", "sharpe", "acceptable"]].round(4).to_string())

gagnant : S0-W3-504|VT15-21-obl
annualise         0.1700
volatilite        0.1453
repli_max        -0.2772
es99              0.0349
sharpe            1.0496
beta_spy          0.7678
centile_sharpe    0.8894

taux d'acceptation par famille de protection :
famille_protection
MX     0.056
P      0.333
PUT    0.208
VT     0.109

regles acceptables sans protection : ['S0-W4-252|P0', 'S1-W2-252|P0', 'S1-W2-504|P0', 'S1-W4-252|P0', 'S1-W4-504|P0', 'S2-W2-252|P0', 'S2-W2-504|P0', 'S2-W4-252|P0', 'S2-W4-504|P0', 'S3-W2-252|P0', 'S3-W4-252|P0', 'S3-W4-504|P0']

centile de Sharpe median des regles, par selection :
selection
S0    0.377
S1    0.135
S2    0.074
S3    0.131

portefeuilles aleatoires au-dessus du Sharpe de T1S (0,8604) : 93.1 %
portefeuilles aleatoires qui satisfont les trois criteres face a T1S : 1.3 %
Sharpe du gagnant face aux portefeuilles aleatoires : superieur a 88.9 % d'entre eux

meilleure regle par famille de protection :
                       annualise  volatilite  repli_m